#Cleaning: "In which country are you located?"
working notebook for standardizing the country column.

In [1]:
#Medeea
import pandas as pd
import numpy as np

In [2]:
#Medeea
df = pd.read_excel('Responses.xlsx')
df.shape

(685, 57)

In [3]:
#Medeea; the column we're cleaning
col2 = "Which description fits your parents' country of origin the best?"

In [4]:
#Medeea;check the raw values
df[col2].value_counts(dropna=False)

Which description fits your parents' country of origin the best?
Both parents are from the EU                                      139
Both parents are from outside the EU                              138
Both parents are born and brought up in the Netherlands            93
Both parents are born and brought up in Germany                    60
Both parents are born and brought up in the Germany                60
One parent is from the EU and one parent is outside of the EU      51
Both parents are born and brought up in the Spain                  36
Both parents are born and brought up in Greece                     36
Both parents are born and brought up in Spain                      34
Both parents are born and brought up in the Greece                 15
Both parents are from EU                                            7
Both parents are from Greece                                        5
NaN                                                                 4
One parent is from Greece

In [5]:
#Medeea;strip whitespace and standardize case
df[col2].astype(str).str.strip()

0      Both parents are born and brought up in the Spain
1      Both parents are born and brought up in the Spain
2      Both parents are born and brought up in the Spain
3      Both parents are born and brought up in the Spain
4      Both parents are born and brought up in the Spain
                             ...                        
680                 Both parents are from outside the EU
681    One parent is from the EU and one parent is ou...
682    Both parents are born and brought up in the Ne...
683    Both parents are born and brought up in the Ne...
684    Both parents are born and brought up in the Ne...
Name: Which description fits your parents' country of origin the best?, Length: 685, dtype: str

In [6]:
#Medeea; This dictionary maps every known variant to one clean label.
mapping = {
    'both parents are from the eu': 'Both parents are from the EU',
    'both parents are from eu':     'Both parents are from the EU',

    'both parents are from outside the eu': 'Both parents are from outside the EU',

    'both parents are born and brought up in the netherlands': 'Both parents are born and brought up in the Netherlands',

    'both parents are born and brought up in germany':     'Both parents are born and brought up in Germany',
    'both parents are born and brought up in the germany': 'Both parents are born and brought up in Germany',

    'both parents are born and brought up in spain':     'Both parents are born and brought up in Spain',
    'both parents are born and brought up in the spain': 'Both parents are born and brought up in Spain',

    'both parents are born and brought up in greece':     'Both parents are born and brought up in Greece',
    'both parents are born and brought up in the greece': 'Both parents are born and brought up in Greece',
    'both parents are from greece':                       'Both parents are born and brought up in Greece',

    'one parent is from the eu and one parent is outside of the eu': 'One parent is from the EU and one parent is outside the EU',
    'one parent is from the eu and one parent is outside the eu':    'One parent is from the EU and one parent is outside the EU',

    'one parent is from greece, and one parent is outside of greece': 'One parent is from Greece and one parent is outside Greece',
}

normalized = df[col2].str.strip().str.lower()   # NaN stays NaN
df[col2] = normalized.map(mapping).fillna(df[col2])

df[col2].value_counts(dropna=False)

Which description fits your parents' country of origin the best?
Both parents are from the EU                                      146
Both parents are from outside the EU                              138
Both parents are born and brought up in Germany                   120
Both parents are born and brought up in the Netherlands            93
Both parents are born and brought up in Spain                      70
One parent is from the EU and one parent is outside the EU         54
Both parents are born and brought up in Greece                     51
Both parents are from Greece                                        5
NaN                                                                 4
One parent is from Greece, and one parent is outside of Greece      4
Name: count, dtype: int64

In [7]:
#Medeea
agree_cols = [
    'I sometimes spend more time scrolling than I had planned',
    'I feel I am in control of my feed',
    'I can influence the type of content I get on my feed',
    'The algorithm decides what I watch',
]

# For every column in the batch: strip spaces, lowercase, translate to a number.
df[agree_cols] = df[agree_cols].apply(lambda s: s.str.strip().str.lower().map({
    'strongly disagree': 1,
    'disagree':          2,
    'neutral':           3,
    'agree':             4,
    'strongly agree':    5,
}))

In [8]:
#Medeea
freq_cols = [
    "I end up watching videos I didn't intend to watch"
]

df[freq_cols] = df[freq_cols].apply(lambda s: s.str.strip().str.lower().map({
    'never':     1,
    'rarely':    2,
    'sometimes': 3,
    'often':     4,
    'always':    5,
}))

In [9]:
#Medeea; revert likert scale for "The algorithm decides what I watch" so that higher numbers indicate more agreement
df['The algorithm decides what I watch'] = 6 - df['The algorithm decides what I watch']

In [10]:
#Medeea; check the values of the cleaned columns
for col in agree_cols + freq_cols:
    print(df[col].value_counts(dropna=False).sort_index(), '\n')

I sometimes spend more time scrolling than I had planned
1.0      5
2.0     38
3.0     66
4.0    298
5.0    270
NaN      8
Name: count, dtype: int64 

I feel I am in control of my feed
1.0     23
2.0    123
3.0    219
4.0    255
5.0     56
NaN      9
Name: count, dtype: int64 

I can influence the type of content I get on my feed
1.0      6
2.0     57
3.0    110
4.0    412
5.0     89
NaN     11
Name: count, dtype: int64 

The algorithm decides what I watch
1.0     88
2.0    374
3.0    146
4.0     62
5.0      7
NaN      8
Name: count, dtype: int64 

I end up watching videos I didn't intend to watch
1.0     14
2.0    107
3.0    278
4.0    217
5.0     58
NaN     11
Name: count, dtype: int64 



In [11]:
#Medeea;One-hot encode the parents' origin column.
#prefix='origin' name prefix for the new columns, e.g. "origin_Both parents are from the EU"
df = pd.get_dummies(df, columns=[col2], prefix='origin', dtype=int)

# Show the new columns
print([c for c in df.columns if c.startswith('origin_')])

['origin_Both parents are born and brought up in Germany', 'origin_Both parents are born and brought up in Greece', 'origin_Both parents are born and brought up in Spain', 'origin_Both parents are born and brought up in the Netherlands', 'origin_Both parents are from outside the EU', 'origin_Both parents are from the EU', 'origin_Both parents are from\xa0Greece', 'origin_One parent is from the EU and one parent is outside the EU', 'origin_One parent is from\xa0Greece, and one parent is outside of\xa0Greece']


In [12]:
#Medeea;save to a new file
df.to_excel('Responses_cleaned.xlsx', index=False)
print('Saved -> Responses_cleaned.xlsx')

Saved -> Responses_cleaned.xlsx
